In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import fdb

In [2]:
config_path = Path("config.json")


def load_config(path: Path) -> dict:
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError as e:
        raise RuntimeError(f"[CONFIG] Файл не найден: {path.resolve()}") from e
    except json.JSONDecodeError as e:
        raise RuntimeError(f"[CONFIG] Ошибка JSON в {path.resolve()}: {e}") from e


def _get_required(cfg: dict, key: str, section_name: str):
    value = cfg.get(key)
    if value is None or (isinstance(value, str) and not value.strip()):
        raise RuntimeError(f"[CONFIG] В секции '{section_name}' отсутствует обязательное поле '{key}'")
    return value


def connect_firebird(db_cfg: dict, section_name: str):
    dsn = _get_required(db_cfg, "dsn", section_name)
    user = _get_required(db_cfg, "user", section_name)
    password = _get_required(db_cfg, "password", section_name)
    try:
        conn = fdb.connect(dsn=dsn, user=user, password=password)
        cursor = conn.cursor()
        print(f"[DB] Подключение '{section_name}' установлено: {dsn}")
        return conn, cursor
    except Exception as e:
        raise RuntimeError(f"[DB] Не удалось подключиться к '{section_name}' ({dsn}): {e}") from e


def run_query(cur, sql: str, ctx: str):
    try:
        cur.execute(sql)
        data = cur.fetchall()
        print(f"[OK] {ctx}: rows={len(data)}")
        return data
    except Exception as e:
        raise RuntimeError(f"[SQL] Ошибка в '{ctx}'. Запрос: {sql}. Причина: {e}") from e


cfg = load_config(config_path)
main_db_cfg = cfg.get("database")
if not isinstance(main_db_cfg, dict):
    raise RuntimeError("[CONFIG] В config.json отсутствует объект 'database'")

# Если отдельной секции tp_database нет, используем те же параметры, что и для database.
tp_db_cfg = cfg.get("tp_database", main_db_cfg)
if not isinstance(tp_db_cfg, dict):
    raise RuntimeError("[CONFIG] Секция 'tp_database' должна быть объектом")

con, cur = connect_firebird(main_db_cfg, "database")
tp_con, tp_cur = connect_firebird(tp_db_cfg, "tp_database")

In [ ]:
# Батчевая выгрузка ALLDATA сразу в CSV (без fetchall)
out_csv = str(Path.cwd() / "ALLDATA.csv")
batch_size = 50_000

try:
    cur.execute("SELECT IDPARAM, IDPROGRAM, FVALUE, TADATE, PLANT, UNIT FROM ALLDATA")
except Exception as e:
    raise RuntimeError(f"[SQL] Не удалось выполнить SELECT из ALLDATA: {e}") from e

first_chunk = True
total_rows = 0

try:
    while True:
        rows = cur.fetchmany(batch_size)
        if not rows:
            break

        chunk_df = pd.DataFrame(rows, columns=[
            "IDPARAM", "IDPROGRAM", "FVALUE", "TADATE", "PLANT", "UNIT"
        ])
        chunk_df.to_csv(out_csv, mode="w" if first_chunk else "a", header=first_chunk, index=False)

        total_rows += len(rows)
        first_chunk = False
except Exception as e:
    raise RuntimeError(f"[EXPORT] Ошибка батчевой выгрузки ALLDATA в {out_csv}: {e}") from e

if total_rows == 0:
    print(f"[WARN] Таблица ALLDATA пуста. CSV создан: {out_csv}")
else:
    print(f"[OK] ALLDATA exported to {out_csv}. Rows: {total_rows}")

In [6]:
# Для быстрого просмотра читаем только первую строку из сохраненного CSV
try:
    ALLDATA_HEAD = pd.read_csv(out_csv, nrows=1)
except FileNotFoundError as e:
    raise RuntimeError(f"[EXPORT] Файл {out_csv} не найден после выгрузки") from e
except Exception as e:
    raise RuntimeError(f"[EXPORT] Не удалось прочитать {out_csv}: {e}") from e

In [7]:
ALLDATA_HEAD.iloc[0]

IDPARAM                        2
IDPROGRAM                      2
FVALUE                 42.200001
TADATE       2025-05-19 15:33:36
PLANT                         L 
UNIT                          C 
Name: 0, dtype: object

In [8]:
#ALLDATA.to_csv('ALLDATA.csv', index=False)

In [9]:
# PARAMETR

parametr = run_query(cur, "SELECT * FROM PARAMETR", "load PARAMETR")
if not parametr:
    raise RuntimeError("[DATA] Таблица PARAMETR пуста")

for i in range(len(parametr[0])):
    print(type(parametr[0][i]), ', ')
print(len(parametr[0]))

<class 'int'> , 
<class 'str'> , 
<class 'float'> , 
<class 'float'> , 
<class 'float'> , 
<class 'float'> , 
<class 'str'> , 
<class 'int'> , 
<class 'int'> , 
<class 'float'> , 
<class 'float'> , 
<class 'float'> , 
<class 'float'> , 
13


In [10]:
entries_parametr = len(parametr)
PARAMETR = pd.DataFrame({'ID' : [parametr[i][0] for i in range(entries_parametr)],
                  'PARAMNAME' : [parametr[i][1] for i in range(entries_parametr)],
                  'MINVALUE' : [parametr[i][2] for i in range(entries_parametr)],
                  'MAXVALUE' : [parametr[i][3] for i in range(entries_parametr)],
                  'MINLIMIT' : [parametr[i][4] for i in range(entries_parametr)],
                  'MAXLIMIT' : [parametr[i][5] for i in range(entries_parametr)],
                  'UNITS' : [parametr[i][6] for i in range(entries_parametr)],
                  'MULTIPLY' : [parametr[i][7] for i in range(entries_parametr)],
                  'ISDRAW' : [parametr[i][8] for i in range(entries_parametr)],
                  'AC' : [parametr[i][9] for i in range(entries_parametr)],
                  'PC' : [parametr[i][10] for i in range(entries_parametr)],
                  'PP' : [parametr[i][11] for i in range(entries_parametr)],
                  'AP' : [parametr[i][12] for i in range(entries_parametr)],
                })

In [11]:
PARAMETR.iloc[0]

ID                                    1
PARAMNAME    Тепловая мощность реактора
MINVALUE                            0.0
MAXVALUE                         4000.0
MINLIMIT                       -10000.0
MAXLIMIT                       -10000.0
UNITS                               МВт
MULTIPLY                              1
ISDRAW                                1
AC                             -0.00001
PC                               2750.0
PP                             -10000.0
AP                             -10000.0
Name: 0, dtype: object

In [12]:
#PARAMETR.to_csv('PARAMETR.csv', index=False)

In [13]:
# PROGRAM

program = run_query(cur, "SELECT * FROM PROGRAM", "load PROGRAM")
if not program:
    raise RuntimeError("[DATA] Таблица PROGRAM пуста")

for i in range(len(program[0])):
    print(type(program[0][i]), ', ')
print(len(program[0]))

<class 'int'> , 
<class 'str'> , 
<class 'str'> , 
<class 'int'> , 
4


In [14]:
entries_program = len(program)
PROGRAM = pd.DataFrame({'ID' : [program[i][0] for i in range(entries_program)],
                  'PROGRAMNAME' : [program[i][1] for i in range(entries_program)],
                  'SHORTNAME' : [program[i][2] for i in range(entries_program)],
                  'ISDRAW' : [program[i][3] for i in range(entries_program)]
                })

In [15]:
PROGRAM.iloc[0]

ID                  1
PROGRAMNAME    Тройка
SHORTNAME         trk
ISDRAW              0
Name: 0, dtype: object

In [16]:
#PROGRAM.to_csv('PROGRAM.csv', index=False)

### ТП (KC2025)

In [17]:
# Подключение tp_database создаётся в ячейке конфигурации (cell 1) из config.json
print("[DB] tp_database cursor ready")

In [18]:
nr = run_query(tp_cur, "SELECT TADATE, KB FROM NR", "load NR")
entries = len(nr)
if entries == 0:
    raise RuntimeError("[DATA] Таблица NR пуста")

NR = pd.DataFrame({'TADATE' : [nr[i][0] for i in range(entries)],
                  'KB' : [nr[i][1] for i in range(entries)]})
#NR.to_csv('NR.csv', index=False)

In [19]:
ci = run_query(tp_cur, "SELECT TADATE, ENVYR FROM CI", "load CI")
entries = len(ci)
if entries == 0:
    raise RuntimeError("[DATA] Таблица CI пуста")

CI = pd.DataFrame({'TADATE' : [ci[i][0] for i in range(entries)],
                  'ENVYR' : [ci[i][1] for i in range(entries)]})
#CI.to_csv('CI.csv', index=False)

In [20]:
cl = run_query(tp_cur, "SELECT TADATE, WTTK FROM CL", "load CL")
entries = len(cl)
if entries == 0:
    raise RuntimeError("[DATA] Таблица CL пуста")

CL = pd.DataFrame({'TADATE' : [cl[i][0] for i in range(entries)],
                  'WTTK' : [cl[i][1] for i in range(entries)]})
#CL.to_csv('CL.csv', index=False)

In [21]:
da = run_query(tp_cur, "SELECT TADATE, CR, RASX FROM DA", "load DA")
entries = len(da)
if entries == 0:
    raise RuntimeError("[DATA] Таблица DA пуста")

DA = pd.DataFrame({'TADATE' : [da[i][0] for i in range(entries)],
                  'CR' : [da[i][1] for i in range(entries)],
                  'RASX' : [da[i][2] for i in range(entries)]})
#DA.to_csv('DA.csv', index=False)

In [22]:
coords = run_query(tp_cur, "SELECT K8 FROM COORDS", "load COORDS.K8")
st = run_query(tp_cur, "SELECT CR FROM COORDS", "load COORDS.CR")

if not coords or coords[0][0] is None:
    raise RuntimeError("[DATA] COORDS.K8 пусто или NULL")
if not st or st[0][0] is None:
    raise RuntimeError("[DATA] COORDS.CR пусто или NULL")

if len(coords[0][0]) < 1884:
    raise RuntimeError(f"[DATA] COORDS.K8 содержит {len(coords[0][0])} элементов, ожидалось >= 1884")
if len(st[0][0]) < 228:
    raise RuntimeError(f"[DATA] COORDS.CR содержит {len(st[0][0])} элементов, ожидалось >= 228")

idx = list(range(1884))
coord_id = pd.Series(coords[0][0], index=idx)
ids = list(range(228))
st_id = pd.Series(st[0][0], index=ids)

In [23]:
class TPoint:
    def __init__(self, X = 0, Y = 0):
        self.X = X
        self.Y = Y

def getCoordLine(Coord8, isBigCartogram):
    NA = [1,11,31,55,83,115,149,185,223,263,305,349,395,443,491,
			541,591,643,695,749,803,857,911,965,1021,1077,1133,1189,1245,1301,
			1357,1413,1469,1525,1579,1633,1687,1741,1795,1847,1899,1949,1999,
			2047,2095,2141,2185,2227,2267,2305,2341,2375,2407,2435,2459,2479,2489]
    NA1 = [1,15,35,59,87,117,149,183,219,257,297,339,381,425,469,
			515,561,607,655,703,751,799,847,895,943,991,1039,1087,1135,1183,
			1231,1279,1325,1371,1417,1461,1505,1547,1589,1629,1667,1703,1737,
			1769,1799,1827,1851,1871,1885]
    Error = False
    L = -1
    NY = Coord8.Y
    NX = Coord8.X
    if isBigCartogram:
        NY = 60 - NY + (NY // 10) * 2
        if (NY <= 0) or (NY >= 57):
            Error = True
        else:
            L = ((NA[NY-1]+NA[NY]) // 2)+NX-(NX // 10)*2-32
            if (L < NA[NY-1]) or (L >= NA[NY]) or (L<=0) or (L>2488):
                Error = True
    else:
        NY = 56-NY+(NY // 10)*2
        if (NY <= 0) or (NY >= 49):
            Error = True
        else:
            L = ((NA1[NY-1]+NA1[NY]) // 2)+NX-(NX // 10)*2-32
            if (L<NA1[NY-1]) or (L>=NA1[NY]) or (L<=0) or (L>1884):
                Error = True
    return L if not Error else -1

def getCoord8(CoordLine, isBigCartogram):
    I = 0
    J = 0
    IB = [24,19,17,15,13,12,11,10,9,8,7,6,5,5,4,4,3,3,
          2,2,2,2,2,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,3,3,4,
          4,5,5,6,7,8,9,10,11,12,13,15,17,19,24]
    IT = [-23,-8,14,40,70,103,138,175,214,255,298,343,
          390,438,487,537,588,640,693,747,801,855,909,964,
          1020,1076,1132,1188,1244,1300,1356,1412,1468,1523,
          1577,1631,1685,1739,1792,1844,1895,1945,1994,2042,
          2089,2134,2177,2218,2257,2294,2329,2362,2392,2418,2440,2455]
    IB1 = [22,19,17,15,14,13,12,11,10,9,8,8,7,7,6,
           6,6,5,5,5,5,5,5,5,5,5,5,5,5,5,5,6,6,6,7,7,8,8,
           9,10,11,12,13,14,15,17,19,22]
    IT1 = [-21,-4,18,44,73,104,137,172,209,248,289,
           331,374,418,463,509,555,602,650,698,746,794,842,890,
           938,986,1034,1082,1130,1178,1226,1273,1319,1365,1410,
           1454,1497,1539,1580,1619,1656,1691,1724,1755,1784,1810,1832,1849]

    result = TPoint(-1, -1)
    if isBigCartogram:
        for II in range(2, 57):
            I = II-1
            if (CoordLine<IB[II-1]+IT[II-1]):
                break
            I = II
        J = CoordLine-IT[I-1]
    else:
        for II in range(2, 49):
            I = II-1
            if (CoordLine<IB1[II-1]+IT1[II-1]):
                break
            I = II
        J = CoordLine - IT1[I-1]
        I = I+4
    
    I = 57-I
    result.Y = ((I+3) // 8)*2+I+3
    result.X = ((J+3) // 8)*2+J+3
    return result

def getLinBigToSmall(CoordLine):
    C8big = getCoord8(CoordLine, True)
    CoordLineSmall = getCoordLine(C8big, False)
    if(CoordLineSmall > 0):
        return getCoord8(CoordLineSmall)
    else:
        return TPoint(-1, -1)

def getC8SmallToBig(Coord8):
    return getCoord8(getCoordLine(Coord8, False), True)

bigcoords = [getC8SmallToBig(getCoord8(i, False)) for i in range(1884)]
bigcoord_id = [("0"+(str(p.X)) if p.X < 10 else str(p.X)) + (("0"+str(p.Y))if p.Y<10 else str(p.Y)) for p in bigcoords]
test_set = set(bigcoord_id)
len(test_set), len(bigcoord_id)

(1884, 1884)

In [24]:
def getChParametr():
    CH_PARAMETR = pd.DataFrame({'ID' : np.arange(110001, 110006),
                      'PARAMNAME' : ['Загрузка реактора для нейтронно-физических программ',
                                     'Энерговыработка ТВС реактора',
                                     'Тепловая мощность в ТК',
                                     'Глубина погружения стержней СУЗ',
                                     'Расходы теплоносителя в ТК и каналах КОСУЗ'],
                      'MINVALUE' : [0] * 5,
                      'MAXVALUE' : [50, 5000, 5, 700, 50],
                      'UNITS' : ['', 'МВт*сут', 'МВт', 'см', 'куб.м./ч'],
                      'MULTIPLY' : [1] * 5,
                      'ISDRAW' : [1] * 5
                    })
    return CH_PARAMETR
    #CH_PARAMETR.to_csv('C:\\Users\\rogat\\Safe\\DB\\CH_PARAMETR_CHS.csv', index=False)


In [31]:
coords_list = [coord_id[i] for i in range(1884)]
st_list = [st_id[i] for i in range(228)]
letter_S_list = ['S'] * 1884
letter_A_list = ['A'] * 1884

big_coord = []
for i in range(1884):
    c = coord_id[i]
    p = TPoint(int(c // 100), int(c % 100))
    mapped = getCoordLine(getC8SmallToBig(p), True)
    if mapped < 0:
        raise RuntimeError(f"[COORD] Не удалось преобразовать coord_id[{i}]={c}")
    big_coord.append(mapped)

big_coord[:10]

[1278, 1230, 1182, 1134, 1086, 1038, 990, 942, 894, 846]

In [32]:
N = len(NR)
len(letter_S_list * 1884)

3549456

In [33]:
#from itertools.chain import from_iterable

def getChData(N, NR, CI, CL, DA):
    try:
        if N <= 0:
            raise ValueError("N должен быть > 0")

        CH_DATA = pd.DataFrame({'IDPARAM' : [110001] * (1884 * N),
                            'COORD' : coords_list * N,
                            'IDPROGRAM': [8] * (1884 * N),
                            'TADATE' : [[NR['TADATE'][j]] for j in range(N) for i in range(1884)],
                            'PLANT' : letter_S_list * N,
                            'UNIT' : letter_A_list * N,
                            'FVALUE' : [NR['KB'][j][big_coord[i]] for j in range(N) for i in range(1884)]})
        print("1 - NR done")

        CH_DATA = pd.concat([CH_DATA,
                pd.DataFrame({'IDPARAM' : [110002] * (1884 * N),
                            'COORD' : coords_list * N,
                            'IDPROGRAM': [8] * (1884 * N),
                            'TADATE' : [CI['TADATE'][j] for j in range(N) for i in range(1884)],
                            'PLANT' : letter_S_list * N,
                            'UNIT' : letter_A_list * N,
                            'FVALUE' : [CI['ENVYR'][j][i] for j in range(N) for i in range(1884)]})
                            ])
        print("2 - CI done")

        CH_DATA = pd.concat([CH_DATA,
                pd.DataFrame({'IDPARAM' : [110003] * (1884 * N),
                            'COORD' : coords_list * N,
                            'IDPROGRAM': [8] * (1884 * N),
                            'TADATE' : [CL['TADATE'][j] for j in range(N) for i in range(1884)],
                            'PLANT' : letter_S_list * N,
                            'UNIT' : letter_A_list * N,
                            'FVALUE' : [CL['WTTK'][j][i] for j in range(N) for i in range(1884)]})
                            ])
        print("3 - CL done")

        CH_DATA = pd.concat([CH_DATA,
                pd.DataFrame({'IDPARAM' : [110004] * (228*N),
                            'COORD' : st_list * N,
                            'IDPROGRAM': [8] * (228 * N),
                            'TADATE' : [DA['TADATE'][j] for j in range(N) for i in range(228)],
                            'PLANT' : letter_S_list[:228] * N,
                            'UNIT' : letter_A_list[:228] * N,
                            'FVALUE' : [DA['CR'][j][i] for j in range(N) for i in range(228)] })
                            ])
        print("4 - DA half done")

        CH_DATA = pd.concat([CH_DATA,
                pd.DataFrame({'IDPARAM' : [110005] * (1884*N),
                            'COORD' : coords_list * N,
                            'IDPROGRAM': [8] * (1884*N),
                            'TADATE' : [DA['TADATE'][j] for j in range(N) for i in range(1884)],
                            'PLANT' : letter_S_list * N,
                            'UNIT' : letter_A_list * N,
                            'FVALUE' : [DA['RASX'][j][i] for j in range(N) for i in range(1884)]})
                            ])
        print("4 - DA done.")

        return CH_DATA
    except KeyError as e:
        raise RuntimeError(f"[CH_DATA] Отсутствует колонка: {e}") from e
    except IndexError as e:
        raise RuntimeError(f"[CH_DATA] Ошибка индексов при формировании CH_DATA: {e}") from e
    except Exception as e:
        raise RuntimeError(f"[CH_DATA] Ошибка формирования CH_DATA: {e}") from e
#CH_DATA.to_csv('C:\\Users\\rogat\\Safe\\DB\\CH_DATA_CHS.csv', index=False)

In [34]:
len(coords_list)

1884

In [35]:
dir_CH_PARAMETR = str(Path.cwd() / 'CH_PARAMETR_CHS.csv')
dir_CH_DATA = str(Path.cwd() / 'CH_DATA_CHS.csv')

try:
    ch_parametr_df = getChParametr()
    ch_parametr_df.to_csv(dir_CH_PARAMETR, index=False)
    print(f"[OK] CH_PARAMETR exported: {dir_CH_PARAMETR}, rows={len(ch_parametr_df)}")
except Exception as e:
    raise RuntimeError(f"[EXPORT] Ошибка сохранения CH_PARAMETR: {e}") from e

try:
    ch_data_df = getChData(len(NR), NR, CI, CL, DA)
    ch_data_df.to_csv(dir_CH_DATA, index=False)
    print(f"[OK] CH_DATA exported: {dir_CH_DATA}, rows={len(ch_data_df)}")
except Exception as e:
    raise RuntimeError(f"[EXPORT] Ошибка сохранения CH_DATA: {e}") from e

1 - NR done
2 - CI done
3 - CL done
4 - DA half done
4 - DA done.


In [ ]:
# Безопасное закрытие ресурсов
for obj_name, obj in [("cur", cur), ("tp_cur", tp_cur), ("con", con), ("tp_con", tp_con)]:
    try:
        obj.close()
    except Exception as e:
        print(f"[WARN] Не удалось закрыть {obj_name}: {e}")

print("[DB] Соединения закрыты")